# 03 — QLoRA Fine-Tuning

Fine-tunes Qwen 2.5-3B-Instruct on the Romanian city recommendation dataset  
using **LoRA adapters** on a 4-bit quantized model (QLoRA).

## What the model learns
- **Conversation behaviour**: how to ask clarifying questions about relocation preferences in Romanian
- **Grounded presentation**: how to synthesize city recommendations from a `[CONTEXT RAG]` block — citing only what's provided, not memorised facts

## Requirements
```bash
pip install torch transformers datasets peft trl bitsandbytes accelerate
```
- Requires a CUDA GPU (8 GB+ VRAM recommended)
- On Windows: use `bitsandbytes-windows` package or run in WSL2
- On CPU-only: set `load_in_4bit=False` and reduce batch size to 1

## Model ID note
"Qwen 3.5-2B" likely maps to `Qwen/Qwen2.5-3B-Instruct` on HuggingFace.  
Check `~/.lmstudio/models/` for the local path if downloading is not possible.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path(".").resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

FINE_TUNING_DIR = REPO_ROOT / "fine_tuning"
TRAIN_FILE = FINE_TUNING_DIR / "data" / "dataset_train.jsonl"
VAL_FILE   = FINE_TUNING_DIR / "data" / "dataset_val.jsonl"

assert TRAIN_FILE.exists(), f"Run 02_prepare_dataset.ipynb first. Missing: {TRAIN_FILE}"
assert VAL_FILE.exists(),   f"Run 02_prepare_dataset.ipynb first. Missing: {VAL_FILE}"

print(f"Train: {TRAIN_FILE}")
print(f"Val  : {VAL_FILE}")

## Step 1 — Load configs

In [ ]:
from fine_tuning.configs.lora_config import (
    DEFAULT_MODEL_ID,
    QWEN_RESPONSE_TEMPLATE,
    ADAPTER_OUTPUT_DIR,
    lora_config,
    quant_config,
    training_config,
)

# Optionally override the model ID with a local path from LM Studio:
# MODEL_ID = r"C:\Users\YourName\.lmstudio\models\Qwen\Qwen2.5-3B-Instruct"
MODEL_ID = DEFAULT_MODEL_ID

print(f"Model           : {MODEL_ID}")
print(f"LoRA rank       : {lora_config.r}")
print(f"4-bit quant     : {quant_config.load_in_4bit}")
print(f"Epochs          : {training_config.num_train_epochs}")
print(f"Effective batch : {training_config.per_device_train_batch_size * training_config.gradient_accumulation_steps}")
print(f"Learning rate   : {training_config.learning_rate}")
print(f"Adapter output  : {ADAPTER_OUTPUT_DIR}")

## Step 2 — Load model with 4-bit quantization

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f"Compute dtype: {compute_dtype}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=quant_config.load_in_4bit,
    bnb_4bit_quant_type=quant_config.bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=quant_config.bnb_4bit_use_double_quant,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"   # Required for causal LM training

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

print(f"Model loaded. Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

## Step 3 — Apply LoRA adapter

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for 4-bit training (enables gradient checkpointing etc.)
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=lora_config.r,
    lora_alpha=lora_config.lora_alpha,
    lora_dropout=lora_config.lora_dropout,
    bias=lora_config.bias,
    task_type=lora_config.task_type,
    target_modules=lora_config.target_modules,
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## Step 4 — Load and format dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={"train": str(TRAIN_FILE), "validation": str(VAL_FILE)},
)

# Apply Qwen's chat template to convert messages → a single formatted string
def apply_chat_template(example):
    messages = example["messages"]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = dataset.map(apply_chat_template, remove_columns=["messages", "metadata"])

print(f"Train size : {len(dataset['train'])}")
print(f"Val size   : {len(dataset['validation'])}")
print("\nSample formatted text (first 400 chars):")
print(dataset["train"][0]["text"][:400])

## Step 5 — Set up response masking

Loss is computed **only on assistant turns**. User messages, system messages,  
and the `[CONTEXT RAG]` block are masked out (label = -100).  
This teaches the model to generate responses, not memorise inputs.

In [ ]:
from trl import DataCollatorForCompletionOnlyLM

# The response_template must match EXACTLY what the tokenizer produces
# for the start of an assistant turn in Qwen's ChatML format.
# Test: tokenizer.apply_chat_template([{"role":"assistant","content":"x"}], tokenize=False)
response_template = QWEN_RESPONSE_TEMPLATE  # "<|im_start|>assistant\n"

collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template,
    tokenizer=tokenizer,
)

print(f"Response template: {response_template!r}")
print("DataCollator configured — loss will only be computed on assistant turns.")

## Step 6 — Train

In [ ]:
from trl import SFTTrainer, SFTConfig

sft_args = SFTConfig(
    output_dir=str(REPO_ROOT / training_config.output_dir),
    num_train_epochs=training_config.num_train_epochs,
    per_device_train_batch_size=training_config.per_device_train_batch_size,
    gradient_accumulation_steps=training_config.gradient_accumulation_steps,
    learning_rate=training_config.learning_rate,
    lr_scheduler_type=training_config.lr_scheduler_type,
    warmup_ratio=training_config.warmup_ratio,
    weight_decay=training_config.weight_decay,
    max_grad_norm=training_config.max_grad_norm,
    optim=training_config.optim,
    max_seq_length=training_config.max_seq_length,
    packing=training_config.packing,
    eval_strategy=training_config.eval_strategy,
    save_strategy=training_config.save_strategy,
    load_best_model_at_end=training_config.load_best_model_at_end,
    metric_for_best_model=training_config.metric_for_best_model,
    logging_steps=training_config.logging_steps,
    report_to=training_config.report_to,
    dataset_text_field="text",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=collator,
    args=sft_args,
)

print("Starting training...")
trainer.train()

## Step 7 — Save the LoRA adapter

In [ ]:
adapter_path = REPO_ROOT / ADAPTER_OUTPUT_DIR
adapter_path.mkdir(parents=True, exist_ok=True)

trainer.model.save_pretrained(str(adapter_path))
tokenizer.save_pretrained(str(adapter_path))

print(f"LoRA adapter saved to: {adapter_path}")
print("\nTo load for inference:")
print("  from peft import PeftModel")
print(f"  model = PeftModel.from_pretrained(base_model, '{adapter_path}')")

## Step 8 — Quick inference test

In [ ]:
from fine_tuning.src.dataset_formatter import SYSTEM_PROMPT

# Test 1: Preference elicitation (no RAG context) — model should ask questions
test_messages_no_context = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": "Bună ziua! Vreau să mă mut din București. Lucrez în IT."},
]

# Test 2: Grounded recommendation (with RAG context) — model should cite only context
rag_context = """[CONTEXT RAG]
Brașov (județul Brașov, municipiu, 237.589 locuitori):
- Transport: cale ferată, drumuri europene E60/E68, aeroport internațional din 2023
- Climă: temperat-continentală, ierni reci, aproape de Munții Bucegi
- Cultură/Turism: festival Cerbul de Aur, trasee montane, viață activă

Sibiu (județul Sibiu, municipiu, 147.245 locuitori):
- Transport: cale ferată spre Cluj și București, aeroport internațional
- Economie: industrie auto (Continental), IT internațional
- Cultură: centru istoric medieval, festivaluri internaționale
[END CONTEXT RAG]"""

test_messages_with_context = [
    {"role": "system",    "content": SYSTEM_PROMPT},
    {"role": "user",      "content": "Vreau să mă mut din București. Lucrez în IT, remote."},
    {"role": "assistant", "content": "Câteva întrebări: preferați un oraș mare sau mic? Importantă natura?"},
    {"role": "user",      "content": "Mic, sub 300k, cu munți. Tren spre București ideal."},
    {"role": "system",    "content": rag_context},
]


def generate_response(messages, max_new_tokens=400):
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


print("=" * 60)
print("TEST 1 — No context (model should ask clarifying questions):")
print("=" * 60)
resp1 = generate_response(test_messages_no_context)
print(resp1)

print("\n" + "=" * 60)
print("TEST 2 — With [CONTEXT RAG] (model should cite only context):")
print("=" * 60)
resp2 = generate_response(test_messages_with_context)
print(resp2)

# Sanity checks
print("\n--- Sanity checks ---")
print(f"Test 1 contains '?' (clarification): {'?' in resp1}")
print(f"Test 2 cites context: {'conform' in resp2.lower() or 'informaț' in resp2.lower() or 'datele' in resp2.lower()}")
# Test 2 should only mention cities present in the RAG context
unexpected = [c for c in ["Cluj", "Timișoara", "Iași"] if c.lower() in resp2.lower()]
print(f"Test 2 mentions unlisted cities: {unexpected or 'none — good!'}")